In [1]:
from pathlib import Path
import shutil
import yaml
import random
from sklearn.model_selection import KFold
import pandas as pd
from collections import Counter
from tqdm import tqdm
from ultralytics import YOLO
import datetime
import numpy as np

In [2]:
# Configuration
DATASET_DIR = Path("data/dataset")
RUNS_DIR = Path("data/runs").resolve()
print(f"Using RUNS_DIR: {RUNS_DIR}")

KFOLD_SPLITS = 5  # 5 folds with ~165 images = ~132 train / ~33 val per fold

Using RUNS_DIR: C:\Users\matth\Desktop\Bike-Fit\notebooks\03_wheel_detection\data\runs


In [3]:
# Step 1: Collect ALL images and labels from single directory
print("Collecting all images and labels...")

images_dir = DATASET_DIR / 'images' / 'all'
labels_dir = DATASET_DIR / 'labels' / 'all'

# Get all images
all_images = sorted(images_dir.glob('*'))
all_images = [f for f in all_images if f.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']]

# Get corresponding labels
all_labels = []
missing_labels = []

for img in all_images:
    label_path = labels_dir / f"{img.stem}.txt"
    if label_path.exists():
        all_labels.append(label_path)
    else:
        missing_labels.append(img)
        print(f"Warning: Missing label for {img.name}")

if missing_labels:
    print(f"\nFound {len(missing_labels)} images without labels. Consider removing them.")
else:
    print(f"All {len(all_images)} images have corresponding labels")

All 165 images have corresponding labels


In [4]:
# Step 2: Generate feature vectors
print("Generating feature vectors...")

index = [label.stem for label in all_labels]
labels_df = pd.DataFrame([], columns=[0], index=index)

for label in all_labels:
    lbl_counter = Counter()
    
    with open(label) as lf:
        lines = lf.readlines()
    
    for line in lines:
        class_id = int(line.split(" ", 1)[0])
        lbl_counter[class_id] += 1
    
    labels_df.loc[label.stem] = lbl_counter.get(0, 0)

labels_df = labels_df.fillna(0.0)

Generating feature vectors...


C:\Users\matth\AppData\Local\Temp\ipykernel_151464\3461043668.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  labels_df = labels_df.fillna(0.0)


In [5]:
# Step 3: Create K-Fold splits
print(f"Creating {KFOLD_SPLITS}-Fold splits...")

random.seed(42)
kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
kfolds = list(kf.split(labels_df))

folds = [f"split_{n}" for n in range(1, KFOLD_SPLITS + 1)]
folds_df = pd.DataFrame(index=index, columns=folds)

for i, (train, val) in enumerate(kfolds, start=1):
    folds_df.loc[labels_df.iloc[train].index, f"split_{i}"] = "train"
    folds_df.loc[labels_df.iloc[val].index, f"split_{i}"] = "val" 

# Calculate distribution
fold_lbl_distrb = pd.DataFrame(index=folds, columns=[0])

for n, (train_indices, val_indices) in enumerate(kfolds, start=1):
    train_totals = labels_df.iloc[train_indices].sum()
    val_totals = labels_df.iloc[val_indices].sum()
    ratio = val_totals / (train_totals + 1e-7)
    fold_lbl_distrb.loc[f"split_{n}"] = ratio
    
    print(f"Fold {n}: {len(train_indices)} train, {len(val_indices)} val "
          f"({len(val_indices)/(len(train_indices)+len(val_indices))*100:.1f}% val)")

Creating 5-Fold splits...
Fold 1: 132 train, 33 val (20.0% val)
Fold 2: 132 train, 33 val (20.0% val)
Fold 3: 132 train, 33 val (20.0% val)
Fold 4: 132 train, 33 val (20.0% val)
Fold 5: 132 train, 33 val (20.0% val)


In [6]:
# Step 4: Create fold directories and YAML files
print("Creating fold directories...")

save_path = DATASET_DIR.parent / f"kfold_{KFOLD_SPLITS}_splits"
save_path.mkdir(parents=True, exist_ok=True)
ds_yamls = []

for split in folds_df.columns:
    split_dir = save_path / split
    split_dir.mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "train" / "labels").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "images").mkdir(parents=True, exist_ok=True)
    (split_dir / "val" / "labels").mkdir(parents=True, exist_ok=True)
    
    dataset_yaml = split_dir / f"{split}_dataset.yaml"
    ds_yamls.append(dataset_yaml)
    
    with open(dataset_yaml, "w") as ds_y:
        yaml.safe_dump(
            {
                "path": split_dir.as_posix(),
                "train": "train",
                "val": "val",
                "names": {0: "wheel"},
            },
            ds_y,
        )

Creating fold directories...


In [7]:
# Step 5: Copy files to fold directories
print("Copying files to fold directories...")

# Create image-label pairs for easy iteration
image_label_pairs = [(img, labels_dir / f"{img.stem}.txt") for img in all_images 
                     if (labels_dir / f"{img.stem}.txt").exists()]

for image, label in tqdm(image_label_pairs, desc="Copying files"):
    for split, k_split in folds_df.loc[image.stem].items():
        img_to_path = save_path / split / k_split / "images"
        lbl_to_path = save_path / split / k_split / "labels"
        
        shutil.copy(image, img_to_path / image.name)
        shutil.copy(label, lbl_to_path / label.name)

Copying files to fold directories...


Copying files: 100%|██████████| 165/165 [00:03<00:00, 50.88it/s]


In [8]:
# Step 6: Save records
print("Saving fold records...")
folds_df.to_csv(save_path / "kfold_datasplit.csv")
fold_lbl_distrb.to_csv(save_path / "kfold_label_distribution.csv")
print(f"Records saved to: {save_path}")

Saving fold records...
Records saved to: data\kfold_5_splits


In [9]:
# Step 7: Train on each fold
print("Starting K-Fold Cross Validation Training:")

results = {}
weights_path = "../../models/yolo/yolo26n-seg.pt"

for k, dataset_yaml in enumerate(ds_yamls):
    print(f"Training Fold {k+1}/{KFOLD_SPLITS}:")
    
    model = YOLO(weights_path)
    
    results[k] = model.train(
        data=str(dataset_yaml),
        epochs=100,
        imgsz=640,
        batch=16,
        patience=20,
        
        # Augmentation
        augment=True,
        mosaic=1.0,
        mixup=0.1,
        degrees=15.0,
        translate=0.2,
        scale=0.5,
        fliplr=0.5,
        flipud=0.0,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        
        # Training settings
        device=0,
        workers=0,
        project=str(RUNS_DIR / "kfold_training"),
        name=f"fold_{k+1}",
        exist_ok=True,
        deterministic=True,
        
        save=True,
        save_period=10,
    )


Starting K-Fold Cross Validation Training:
Training Fold 1/5:
New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\kfold_5_splits\split_1\split_1_dataset.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixu

In [11]:
# Step 8: Aggregate results
print("K-Fold Cross Validation Results:")

metrics_summary = []

for k in range(KFOLD_SPLITS):
    best_model_path = RUNS_DIR /"kfold_training" / f"fold_{k+1}" / "weights" / "best.pt"
    model = YOLO(best_model_path)
    metrics = model.val(
        data=str(ds_yamls[k]),
        project=str(RUNS_DIR / "kfold_validation"),
        name=f"fold_{k+1}",
        exist_ok=True
    )
    
    fold_metrics = {
        'fold': k+1,
        'box_map50_95': metrics.box.map,
        'box_map50': metrics.box.map50,
        'seg_map50_95': metrics.seg.map,
        'seg_map50': metrics.seg.map50,
    }
    metrics_summary.append(fold_metrics)
    
    print(f"\nFold {k+1}:")
    print(f"  Box mAP@50-95: {metrics.box.map:.3f}")
    print(f"  Box mAP@50:    {metrics.box.map50:.3f}")
    print(f"  Seg mAP@50-95: {metrics.seg.map:.3f}")
    print(f"  Seg mAP@50:    {metrics.seg.map50:.3f}")

metrics_df = pd.DataFrame(metrics_summary)
print(f"\n{'='*50}")
print("Average Performance (Mean ± Std):")
print(f"{'='*50}")
print(f"Box mAP@50-95: {metrics_df['box_map50_95'].mean():.3f} ± {metrics_df['box_map50_95'].std():.3f}")
print(f"Box mAP@50:    {metrics_df['box_map50'].mean():.3f} ± {metrics_df['box_map50'].std():.3f}")
print(f"Seg mAP@50-95: {metrics_df['seg_map50_95'].mean():.3f} ± {metrics_df['seg_map50_95'].std():.3f}")
print(f"Seg mAP@50:    {metrics_df['seg_map50'].mean():.3f} ± {metrics_df['seg_map50'].std():.3f}")

metrics_df.to_csv(save_path / "kfold_metrics_summary.csv", index=False)
print(f"\nResults saved to: {save_path}")

K-Fold Cross Validation Results:
Ultralytics 8.4.14  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
YOLO26n-seg summary (fused): 139 layers, 2,689,079 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 521.9424.5 MB/s, size: 203.6 KB)
val: Scanning C:\Users\matth\Desktop\Bike-Fit\notebooks\03_wheel_detection\data\kfold_5_splits\split_1\val\labels.cache... 33 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 33/33  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.8s/it 8.5s1.8s0s
                   all         33         44      0.996          1      0.995      0.981      0.996          1      0.995      0.984
Speed: 23.1ms preprocess, 41.7ms inference, 0.0ms loss, 1.3ms postprocess per image
Results saved to C:\Users\matth\Desktop\Bike-Fit\notebooks\03_wheel_detection\data\runs\kfold_validation\fold_1

In [12]:
# After K-Fold completes, find average best epoch
best_epochs = []

for k in range(KFOLD_SPLITS):
    # Load training results
    results_csv = RUNS_DIR / "kfold_training" / f"fold_{k+1}" / "results.csv"
    results_data = pd.read_csv(results_csv)
    
    # Find epoch with best val mAP
    best_epoch = results_data['metrics/mAP50-95(M)'].idxmax()  # Adjust column name as needed
    best_epochs.append(best_epoch)
    print(f"Fold {k+1} best epoch: {best_epoch}")

avg_best_epoch = int(np.mean(best_epochs))
print(f"\nAverage best epoch across folds: {avg_best_epoch}")

Fold 1 best epoch: 85
Fold 2 best epoch: 52
Fold 3 best epoch: 27
Fold 4 best epoch: 89
Fold 5 best epoch: 38

Average best epoch across folds: 58


In [13]:
# Create a data.yaml for all data (no split)
final_data_yaml = save_path / "final_all_data.yaml"

with open(final_data_yaml, "w") as f:
    yaml.safe_dump(
        {
            "path": (DATASET_DIR).as_posix(),
            "train": "images/all",  # ALL images for training
            "val": "images/all",    # Use same for "validation" (won't affect training)
            "names": {0: "wheel"},
        },
        f,
    )

# Train final model with same hyperparameters that worked in K-Fold
model_final = YOLO(weights_path)

results_final = model_final.train(
    data=str(final_data_yaml),
    epochs=avg_best_epoch,  # Use average best epoch from K-Fold
    imgsz=640,
    batch=16,
    patience=0,
    
    # Same augmentation as K-Fold
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    degrees=15.0,
    translate=0.2,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    
    # Training settings
    device=0,
    workers=0,
    project=str(RUNS_DIR / "final_model"),
    name="all_data_deployment",
    exist_ok=True,
    deterministic=True,
    
    save=True,
    save_period=10,
)

print("\nFinal deployment model trained!")
print(f"Model saved to: {RUNS_DIR / 'final_model' / 'all_data_deployment' / 'weights' / 'best.pt'}")

New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.14  Python-3.11.9 torch-2.10.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060 Laptop GPU, 8188MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data\kfold_5_splits\final_all_data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=58, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=../../models/yolo/yolo26n-seg.pt, momentum=0.93